# 🚀 Урок 1. Кто такой Data Analyst в 2027 году: от написания кода к AI-оркестрации

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_00_onboarding/lesson_01_what_is_data_analyst.ipynb)

## 📌 Бизнес-контекст и методология
В 2027 году ценность аналитика больше не измеряется скоростью набора синтаксиса `pd.read_csv` или умением вручную писать вложенные циклы. LLM-модели (Claude 3.7/4.6, GPT-5, Gemini 2.5) генерируют рутинный код за секунды.

**Ключевой сдвиг профессии:**
1. **Problem Framing (Постановка задачи):** Перевод абстрактных болей бизнеса («почему падают продажи?») в математические метрики (AOV, LTV, CR по когортам).
2. **AI-Orchestration & Tool Use:** Формулирование ТЗ для AI-ассистентов, генерация запросов через Cursor/Copilot, анализ неструктурированных данных.
3. **Критический аудит и фактчекинг:** Выявление скрытых галлюцинаций LLM, проверка краевых условий (NULL, дубликаты, сдвиг часовых поясов).
4. **Data Storytelling & Minto Pyramid:** Защита инсайтов перед топ-менеджментом на языке денег, а не технических терминов.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

db_path = ensure_dataset("ecommerce.db")
conn = sqlite3.connect(db_path)
print(f"🟢 Подключение к базе данных успешно установлено: {db_path}")


🟢 Подключение к базе данных успешно установлено: course\datasets\ecommerce.db


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Пример 2027: Декомпозиция бизнес-метрики выручки (Revenue = Traffic * CR * AOV)
traffic = 150000       # Визитов в месяц
conversion_rate = 0.024 # Конверсия в покупку (2.4%)
aov = 3450             # Средний чек (руб.)

revenue = traffic * conversion_rate * aov
print(f"📊 Текущая выручка: {revenue:,.0f} руб.")

# Моделирование сценариев роста: что даст больший прирост выручки?
rev_a = traffic * (conversion_rate + 0.003) * aov
rev_b = traffic * conversion_rate * (aov * 1.10)

print(f"📈 Сценарий A (+0.3% CR): {rev_a:,.0f} руб. (Прирост: +{rev_a - revenue:,.0f} руб.)")
print(f"📈 Сценарий B (+10% AOV): {rev_b:,.0f} руб. (Прирост: +{rev_b - revenue:,.0f} руб.)")


📊 Текущая выручка: 12,420,000 руб.
📈 Сценарий A (+0.3% CR): 13,972,500 руб. (Прирост: +1,552,500 руб.)
📈 Сценарий B (+10% AOV): 13,662,000 руб. (Прирост: +1,242,000 руб.)


### 🎯 Практическое задание 1: Оценка уязвимости воронки
Руководство планирует запустить скидочную акцию, которая поднимет конверсию с 2.4% до 3.0%, но снизит средний чек на 15% из-за промокодов.
Напишите расчет и определите: выгодна ли эта акция компании при том же трафике (150 000 визитов)?

In [3]:
### ВАШ КОД ЗДЕСЬ
traffic = 150000
new_cr = 0.030
new_aov = 3450 * 0.85

promo_revenue = traffic * new_cr * new_aov
base_revenue = 150000 * 0.024 * 3450
diff = promo_revenue - base_revenue

print(f"Выручка по акции: {promo_revenue:,.0f} руб.")
print(f"Разница с базовой: {diff:,.0f} руб.")

assert promo_revenue > 0, "Выручка должна быть больше нуля"
assert diff > 0, "Акция выгодна: прирост положительный!"
print("✅ Тест успешно пройден: акция экономически оправдана!")


Выручка по акции: 13,196,250 руб.
Разница с базовой: 776,250 руб.
✅ Тест успешно пройден: акция экономически оправдана!


### 💡 Промпт для AI-ассистента (Problem Framing)
> *«Действуй как Lead Product Analyst. У меня есть e-commerce воронка со следующими вводными: Трафик = 150к, CR = 2.4%, AOV = 3450 руб. Мы тестируем скидку 15%, ожидаем рост CR до 3.0%. Рассчитай точку безубыточности (Break-Even CR), при которой выручка не упадет, и выдели 3 риска этой акции (каннибализация, отток лояльных, задержка повторных заказов). Выведи ответ по методологии пирамиды Минто.»*